# 05 Regression Models V3 — Time-Series Dataset

This notebook is the regression notebook for the dataset created by **Notebook 4A Time-Series Feature Engineering**.

It automatically loads:

`final_meal_centric_regression_dataset_TIME_SERIES.csv`

Models:
- Linear Regression
- Ridge
- Random Forest
- Gradient Boosting
- XGBoost
- CatBoost, if installed

Targets:
- glucose_60m
- glucose_90m
- glucose_120m
- peak_glucose_2h
- delta_peak_2h
- auc_2h


In [9]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False

PROJECT_PATH = r"C:\Users\Shabnam\Meal_Centric_Glucose_Dissertation"

FEATURE_PATH = os.path.join(PROJECT_PATH, "03_feature_engineered_dataset")
MODEL_PATH = os.path.join(PROJECT_PATH, "05_regression_models_time_series")
PLOT_PATH = os.path.join(MODEL_PATH, "plots")
PRED_PATH = os.path.join(MODEL_PATH, "predictions")
IMPORTANCE_PATH = os.path.join(MODEL_PATH, "feature_importance")

for p in [MODEL_PATH, PLOT_PATH, PRED_PATH, IMPORTANCE_PATH]:
    os.makedirs(p, exist_ok=True)

TIME_SERIES_DATASET = os.path.join(
    FEATURE_PATH,
    "final_meal_centric_regression_dataset_TIME_SERIES.csv"
)

print("XGBoost available:", XGB_AVAILABLE)
print("CatBoost available:", CATBOOST_AVAILABLE)
print("Time-series dataset:", TIME_SERIES_DATASET)


XGBoost available: True
CatBoost available: True
Time-series dataset: C:\Users\Shabnam\Meal_Centric_Glucose_Dissertation\03_feature_engineered_dataset\final_meal_centric_regression_dataset_TIME_SERIES.csv


In [10]:
if not os.path.exists(TIME_SERIES_DATASET):
    raise FileNotFoundError(
        "Time-series dataset not found. Run Notebook 04A first."
    )

DATA_FILE = TIME_SERIES_DATASET
DATASET_VERSION = "time_series_features"

df = pd.read_csv(DATA_FILE)
df = df.loc[:, ~df.columns.duplicated()].copy()
df = df.replace([np.inf, -np.inf], np.nan)

if "participant_id" in df.columns:
    df["participant_id"] = df["participant_id"].astype(str).str.strip()

if "meal_timestamp" in df.columns:
    df["meal_timestamp"] = pd.to_datetime(df["meal_timestamp"], errors="coerce", utc=True)

print("Using dataset:", DATA_FILE)
print("Dataset version:", DATASET_VERSION)
print("Shape:", df.shape)
print("Participants:", df["participant_id"].nunique() if "participant_id" in df.columns else "N/A")
display(df.head())


Using dataset: C:\Users\Shabnam\Meal_Centric_Glucose_Dissertation\03_feature_engineered_dataset\final_meal_centric_regression_dataset_TIME_SERIES.csv
Dataset version: time_series_features
Shape: (8470, 468)
Participants: 96


,participant_id,id,meal_timestamp,meal_item_count,unique_category_count,unique_subcategory_count,dominant_category,dominant_subcategory,dominant_cooking_style,sweets_count,...,ts_day_of_week,ts_is_weekend,ts_is_morning,ts_is_afternoon,ts_is_evening,ts_is_night,ts_hour_sin,ts_hour_cos,ts_dow_sin,ts_dow_cos
0,A4F_10021,A4F_10021_0000.jpg,2022-06-08 13:49:02+00:00,1,1,1,Cereals and Legumes,Bread,Fried,0,...,2,0,0,1,0,0,-0.258819,-0.965926,0.974928,-0.222521
1,A4F_10021,A4F_10021_0003.jpg,2022-06-08 13:49:02+00:00,1,1,1,Dairy and Plant-Based Drinks,Yogurt and Fresh Cheese,Unknown,0,...,2,0,0,1,0,0,-0.258819,-0.965926,0.974928,-0.222521
2,A4F_10021,A4F_10021_0003.jpg,2022-06-08 13:49:02+00:00,1,1,1,Dairy and Plant-Based Drinks,Yogurt and Fresh Cheese,Unknown,0,...,2,0,0,1,0,0,-0.258819,-0.965926,0.974928,-0.222521
3,A4F_10021,A4F_10021_0003.jpg,2022-06-08 13:49:02+00:00,1,1,1,Dairy and Plant-Based Drinks,Yogurt and Fresh Cheese,Unknown,0,...,2,0,0,1,0,0,-0.258819,-0.965926,0.974928,-0.222521
4,A4F_10021,A4F_10021_0003.jpg,2022-06-08 13:49:02+00:00,1,1,1,Dairy and Plant-Based Drinks,Yogurt and Fresh Cheese,Unknown,0,...,2,0,0,1,0,0,-0.258819,-0.965926,0.974928,-0.222521


In [11]:
TARGETS = [
    "glucose_60m",
    "glucose_90m",
    "glucose_120m",
    "peak_glucose_2h",
    "delta_peak_2h",
    "auc_2h"
]

TARGETS = [t for t in TARGETS if t in df.columns]

DROP_COLS = [
    "participant_id", "id", "meal_id", "meal_timestamp", "date",
    "image_path", "participant_folder", "image_exists", "image_path_x", "image_path_y",

    # leakage / future outcome columns
    "glucose_60m", "glucose_90m", "glucose_120m",
    "peak_glucose_2h", "delta_peak_2h", "mean_glucose_2h",
    "auc_2h", "peak_delay_min",

    # text columns
    "food_description", "image_food_description", "nutrition_reasoning_short",
    "nutrition_reasoning_short_x", "nutrition_reasoning_short_y"
]

print("Targets:", TARGETS)


Targets: ['glucose_60m', 'glucose_90m', 'glucose_120m', 'peak_glucose_2h', 'delta_peak_2h', 'auc_2h']


In [12]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


def clean_categorical_columns(X):
    X = X.copy()
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].fillna("Unknown").astype(str)
    return X


def remove_bad_columns(X, missing_threshold=0.70):
    X = X.copy()

    missing_rate = X.isnull().mean()
    high_missing_cols = missing_rate[missing_rate > missing_threshold].index.tolist()
    X = X.drop(columns=high_missing_cols, errors="ignore")

    constant_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
    X = X.drop(columns=constant_cols, errors="ignore")

    return X, high_missing_cols, constant_cols


def prepare_X_y(data, target):
    data = data.copy()
    data = data.dropna(subset=[target]).copy()

    if "meal_timestamp" in data.columns:
        data["meal_timestamp"] = pd.to_datetime(data["meal_timestamp"], errors="coerce", utc=True)
        if "participant_id" in data.columns:
            data = data.sort_values(["participant_id", "meal_timestamp"]).reset_index(drop=True)
        else:
            data = data.sort_values(["meal_timestamp"]).reset_index(drop=True)

    drop_cols = [c for c in DROP_COLS if c in data.columns and c != target]

    X = data.drop(columns=drop_cols, errors="ignore")
    y = data[target].copy()

    if target in X.columns:
        X = X.drop(columns=[target])

    X, high_missing_cols, constant_cols = remove_bad_columns(X, missing_threshold=0.70)
    X = clean_categorical_columns(X)

    return X, y, data, high_missing_cols, constant_cols


def build_preprocessor(X, scale_numeric=False):
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    if scale_numeric:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])

    return preprocessor, numeric_cols, categorical_cols


def temporal_split(X, y, data, test_size=0.20):
    split_idx = int(len(data) * (1 - test_size))

    return (
        X.iloc[:split_idx].copy(),
        X.iloc[split_idx:].copy(),
        y.iloc[:split_idx].copy(),
        y.iloc[split_idx:].copy(),
        data.iloc[:split_idx].copy(),
        data.iloc[split_idx:].copy()
    )


def make_models():
    models = {
        "Linear Regression": {
            "model": LinearRegression(),
            "scale": True
        },
        "Ridge": {
            "model": Ridge(alpha=1.0),
            "scale": True
        },
        "Random Forest": {
            "model": RandomForestRegressor(
                n_estimators=300,
                max_depth=20,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1
            ),
            "scale": False
        },
        "Gradient Boosting": {
            "model": GradientBoostingRegressor(
                n_estimators=250,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ),
            "scale": False
        }
    }

    if XGB_AVAILABLE:
        models["XGBoost"] = {
            "model": XGBRegressor(
                n_estimators=400,
                learning_rate=0.03,
                max_depth=4,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="reg:squarederror",
                random_state=42
            ),
            "scale": False
        }

    if CATBOOST_AVAILABLE:
        models["CatBoost"] = {
            "model": CatBoostRegressor(
                iterations=500,
                depth=6,
                learning_rate=0.04,
                loss_function="RMSE",
                random_seed=42,
                verbose=0
            ),
            "scale": False
        }

    return models


def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = list(numeric_cols)

    try:
        encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        feature_names.extend(list(encoder.get_feature_names_out(categorical_cols)))
    except Exception:
        pass

    return feature_names


def plot_pred_vs_actual(y_true, y_pred, title, path):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.5)

    mn = min(np.min(y_true), np.min(y_pred))
    mx = max(np.max(y_true), np.max(y_pred))

    plt.plot([mn, mx], [mn, mx])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def plot_residuals(y_true, y_pred, title, path):
    residuals = y_true - y_pred

    plt.figure(figsize=(7, 4))
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


In [13]:
all_results = []
best_models = []
feature_log = []

models_config = make_models()

for target in TARGETS:
    print("\n" + "="*80)
    print("TARGET:", target)
    print("="*80)

    X, y, data_used, high_missing_cols, constant_cols = prepare_X_y(df, target)

    print("Samples:", X.shape[0])
    print("Features:", X.shape[1])
    print("High-missing removed:", len(high_missing_cols))
    print("Constant removed:", len(constant_cols))

    feature_log.append({
        "target": target,
        "samples": X.shape[0],
        "features": X.shape[1],
        "high_missing_removed": len(high_missing_cols),
        "constant_removed": len(constant_cols)
    })

    X_train, X_test, y_train, y_test, data_train, data_test = temporal_split(
        X, y, data_used, test_size=0.20
    )

    target_model_objects = {}

    for model_name, cfg in models_config.items():
        print("Training:", model_name)

        preprocessor, numeric_cols, categorical_cols = build_preprocessor(
            X_train,
            scale_numeric=cfg["scale"]
        )

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", cfg["model"])
        ])

        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        metrics = regression_metrics(y_test, preds)

        all_results.append({
            "dataset_version": DATASET_VERSION,
            "target": target,
            "model": model_name,
            "train_samples": len(X_train),
            "test_samples": len(X_test),
            "features": X.shape[1],
            **metrics
        })

        pred_df = pd.DataFrame({
            "target": target,
            "model": model_name,
            "actual": y_test.values,
            "predicted": preds
        })

        if "participant_id" in data_test.columns:
            pred_df["participant_id"] = data_test["participant_id"].values

        if "meal_timestamp" in data_test.columns:
            pred_df["meal_timestamp"] = data_test["meal_timestamp"].astype(str).values

        pred_df.to_csv(
            os.path.join(PRED_PATH, f"predictions_{target}_{model_name.replace(' ', '_')}.csv"),
            index=False
        )

        plot_pred_vs_actual(
            y_test.values,
            preds,
            f"{target} - {model_name}",
            os.path.join(PLOT_PATH, f"{target}_{model_name.replace(' ', '_')}_pred_vs_actual.png")
        )

        plot_residuals(
            y_test.values,
            preds,
            f"{target} - {model_name} Residuals",
            os.path.join(PLOT_PATH, f"{target}_{model_name.replace(' ', '_')}_residuals.png")
        )

        target_model_objects[model_name] = {
            "pipeline": pipe,
            "metrics": metrics,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
            "X_columns": X.columns.tolist(),
            "high_missing_cols": high_missing_cols,
            "constant_cols": constant_cols
        }

    target_results = pd.DataFrame([r for r in all_results if r["target"] == target]).sort_values("RMSE")
    best_name = target_results.iloc[0]["model"]
    best_bundle = target_model_objects[best_name]

    best_models.append({
        "target": target,
        "best_model": best_name,
        **best_bundle["metrics"],
        "bundle": best_bundle
    })

    print("Best model:", best_name)
    display(target_results)

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(MODEL_PATH, "temporal_test_regression_results.csv"), index=False)

feature_log_df = pd.DataFrame(feature_log)
feature_log_df.to_csv(os.path.join(MODEL_PATH, "feature_log.csv"), index=False)

display(results_df.sort_values(["target", "RMSE"]))



TARGET: glucose_60m
Samples: 8372
Features: 419
High-missing removed: 24
Constant removed: 12
Training: Linear Regression
Training: Ridge
Training: Random Forest
Training: Gradient Boosting
Training: XGBoost
Training: CatBoost
Best model: XGBoost


,dataset_version,target,model,train_samples,test_samples,features,MAE,RMSE,R2
4,time_series_features,glucose_60m,XGBoost,6697,1675,419,6.416120,12.296146,0.583307
2,time_series_features,glucose_60m,Random Forest,6697,1675,419,6.156837,12.412038,0.575415
3,time_series_features,glucose_60m,Gradient Boosting,6697,1675,419,7.269136,12.687546,0.556357
5,time_series_features,glucose_60m,CatBoost,6697,1675,419,7.120992,12.951516,0.537704
1,time_series_features,glucose_60m,Ridge,6697,1675,419,12.860969,19.891938,-0.090518
0,time_series_features,glucose_60m,Linear Regression,6697,1675,419,13.248384,20.348861,-0.141192



TARGET: glucose_90m
Samples: 8248
Features: 419
High-missing removed: 24
Constant removed: 12
Training: Linear Regression
Training: Ridge
Training: Random Forest
Training: Gradient Boosting
Training: XGBoost
Training: CatBoost
Best model: Random Forest


,dataset_version,target,model,train_samples,test_samples,features,MAE,RMSE,R2
2,time_series_features,glucose_90m,Random Forest,6598,1650,419,9.333785,12.523727,0.393645
3,time_series_features,glucose_90m,Gradient Boosting,6598,1650,419,9.070322,12.749128,0.371622
4,time_series_features,glucose_90m,XGBoost,6598,1650,419,9.691197,13.076204,0.338967
5,time_series_features,glucose_90m,CatBoost,6598,1650,419,9.898265,13.276891,0.318521
1,time_series_features,glucose_90m,Ridge,6598,1650,419,13.943299,17.900331,-0.238745
0,time_series_features,glucose_90m,Linear Regression,6598,1650,419,14.436991,18.727939,-0.355938



TARGET: glucose_120m
Samples: 8105
Features: 419
High-missing removed: 24
Constant removed: 12
Training: Linear Regression
Training: Ridge
Training: Random Forest
Training: Gradient Boosting
Training: XGBoost
Training: CatBoost
Best model: CatBoost


...


TARGET: delta_peak_2h
Samples: 8470
Features: 419
High-missing removed: 24
Constant removed: 12
Training: Linear Regression
Training: Ridge
Training: Random Forest
Training: Gradient Boosting
Training: XGBoost
Training: CatBoost
Best model: XGBoost


,dataset_version,target,model,train_samples,test_samples,features,MAE,RMSE,R2
4,time_series_features,delta_peak_2h,XGBoost,6776,1694,419,6.771078,10.987536,0.756399
2,time_series_features,delta_peak_2h,Random Forest,6776,1694,419,5.943221,11.124151,0.750303
3,time_series_features,delta_peak_2h,Gradient Boosting,6776,1694,419,7.740363,11.683421,0.724565
5,time_series_features,delta_peak_2h,CatBoost,6776,1694,419,7.588965,11.939238,0.712371
1,time_series_features,delta_peak_2h,Ridge,6776,1694,419,11.672149,18.835031,0.284167
0,time_series_features,delta_peak_2h,Linear Regression,6776,1694,419,11.828467,19.049866,0.267744



TARGET: auc_2h
Samples: 8470
Features: 419
High-missing removed: 24
Constant removed: 12
Training: Linear Regression
Training: Ridge
Training: Random Forest
Training: Gradient Boosting
Training: XGBoost
Training: CatBoost
Best model: Random Forest


,dataset_version,target,model,train_samples,test_samples,features,MAE,RMSE,R2
2,time_series_features,auc_2h,Random Forest,6776,1694,419,479.910825,900.209086,0.769779
4,time_series_features,auc_2h,XGBoost,6776,1694,419,565.983725,946.629734,0.745423
3,time_series_features,auc_2h,Gradient Boosting,6776,1694,419,615.979712,968.642299,0.733446
5,time_series_features,auc_2h,CatBoost,6776,1694,419,706.682291,992.640642,0.720074
1,time_series_features,auc_2h,Ridge,6776,1694,419,916.546124,1458.151185,0.395962
0,time_series_features,auc_2h,Linear Regression,6776,1694,419,927.695064,1464.249333,0.390899


,dataset_version,target,model,train_samples,test_samples,features,MAE,RMSE,R2
32,time_series_features,auc_2h,Random Forest,6776,1694,419,479.910825,900.209086,0.769779
34,time_series_features,auc_2h,XGBoost,6776,1694,419,565.983725,946.629734,0.745423
33,time_series_features,auc_2h,Gradient Boosting,6776,1694,419,615.979712,968.642299,0.733446
35,time_series_features,auc_2h,CatBoost,6776,1694,419,706.682291,992.640642,0.720074
31,time_series_features,auc_2h,Ridge,6776,1694,419,916.546124,1458.151185,0.395962
30,time_series_features,auc_2h,Linear Regression,6776,1694,419,927.695064,1464.249333,0.390899
28,time_series_features,delta_peak_2h,XGBoost,6776,1694,419,6.771078,10.987536,0.756399
26,time_series_features,delta_peak_2h,Random Forest,6776,1694,419,5.943221,11.124151,0.750303
27,time_series_features,delta_peak_2h,Gradient Boosting,6776,1694,419,7.740363,11.683421,0.724565
29,time_series_features,delta_peak_2h,CatBoost,6776,1694,419,7.588965,11.939238,0.712371


In [14]:
cv_results = []
models_config = make_models()

for target in TARGETS:
    print("\nCV target:", target)

    X, y, data_used, high_missing_cols, constant_cols = prepare_X_y(df, target)

    for model_name, cfg in models_config.items():
        print("CV:", model_name)

        preprocessor, numeric_cols, categorical_cols = build_preprocessor(
            X,
            scale_numeric=cfg["scale"]
        )

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", cfg["model"])
        ])

        scoring = {
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        }

        kf = KFold(n_splits=5, shuffle=True, random_state=42)

        try:
            scores = cross_validate(
                pipe,
                X,
                y,
                cv=kf,
                scoring=scoring,
                n_jobs=-1
            )

            cv_results.append({
                "dataset_version": DATASET_VERSION,
                "target": target,
                "model": model_name,
                "MAE_mean": -scores["test_MAE"].mean(),
                "MAE_std": scores["test_MAE"].std(),
                "RMSE_mean": -scores["test_RMSE"].mean(),
                "RMSE_std": scores["test_RMSE"].std(),
                "R2_mean": scores["test_R2"].mean(),
                "R2_std": scores["test_R2"].std(),
                "samples": X.shape[0],
                "features": X.shape[1]
            })

        except Exception as e:
            print("CV failed:", target, model_name, e)

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.to_csv(os.path.join(MODEL_PATH, "cross_validation_results.csv"), index=False)

display(cv_results_df.sort_values(["target", "RMSE_mean"]))



CV target: glucose_60m
CV: Linear Regression
CV: Ridge
CV: Random Forest
CV: Gradient Boosting
CV: XGBoost
CV: CatBoost

CV target: glucose_90m
CV: Linear Regression
CV: Ridge
CV: Random Forest
CV: Gradient Boosting
CV: XGBoost
CV: CatBoost

CV target: glucose_120m
CV: Linear Regression
CV: Ridge
CV: Random Forest
CV: Gradient Boosting
CV: XGBoost
CV: CatBoost

CV target: peak_glucose_2h
CV: Linear Regression
CV: Ridge
CV: Random Forest
CV: Gradient Boosting
CV: XGBoost
CV: CatBoost

CV target: delta_peak_2h
CV: Linear Regression
CV: Ridge
CV: Random Forest
CV: Gradient Boosting
CV: XGBoost
CV: CatBoost

CV target: auc_2h
CV: Linear Regression
CV: Ridge
CV: Random Forest
CV: Gradient Boosting
CV: XGBoost
CV: CatBoost


,dataset_version,target,model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,samples,features
35,time_series_features,auc_2h,CatBoost,527.541188,20.209405,844.770183,46.362234,0.803665,0.020023,8470,419
34,time_series_features,auc_2h,XGBoost,536.032680,17.793779,860.159082,42.358805,0.796602,0.017931,8470,419
32,time_series_features,auc_2h,Random Forest,522.555619,28.479259,910.705352,53.342144,0.771968,0.023078,8470,419
33,time_series_features,auc_2h,Gradient Boosting,589.736350,21.852232,914.487443,46.390482,0.769977,0.021831,8470,419
31,time_series_features,auc_2h,Ridge,704.508183,18.426760,1014.702435,34.653100,0.717019,0.019463,8470,419
30,time_series_features,auc_2h,Linear Regression,706.958768,17.857947,1018.607559,34.323596,0.714858,0.019196,8470,419
29,time_series_features,delta_peak_2h,CatBoost,6.904209,0.251367,11.415593,0.587086,0.774267,0.018620,8470,419
28,time_series_features,delta_peak_2h,XGBoost,7.048023,0.219051,11.637209,0.548163,0.765502,0.017013,8470,419
27,time_series_features,delta_peak_2h,Gradient Boosting,7.730586,0.272852,12.197263,0.583652,0.742331,0.019672,8470,419
26,time_series_features,delta_peak_2h,Random Forest,6.862534,0.234156,12.232846,0.474797,0.740966,0.014711,8470,419


In [15]:
summary_rows = []

for item in best_models:
    target = item["target"]
    best_model_name = item["best_model"]
    bundle = item["bundle"]
    pipe = bundle["pipeline"]

    save_file = os.path.join(MODEL_PATH, f"best_model_{target}.pkl")

    joblib.dump({
        "target": target,
        "best_model_name": best_model_name,
        "dataset_version": DATASET_VERSION,
        "pipeline": pipe,
        "metrics": bundle["metrics"],
        "numeric_cols": bundle["numeric_cols"],
        "categorical_cols": bundle["categorical_cols"],
        "X_columns": bundle["X_columns"],
        "high_missing_cols": bundle["high_missing_cols"],
        "constant_cols": bundle["constant_cols"]
    }, save_file)

    summary_rows.append({
        "target": target,
        "best_model": best_model_name,
        **bundle["metrics"],
        "model_file": save_file
    })

    model = pipe.named_steps["model"]

    if hasattr(model, "feature_importances_"):
        preprocessor = pipe.named_steps["preprocessor"]
        feature_names = get_feature_names(
            preprocessor,
            bundle["numeric_cols"],
            bundle["categorical_cols"]
        )

        n = min(len(feature_names), len(model.feature_importances_))

        imp_df = pd.DataFrame({
            "target": target,
            "feature": feature_names[:n],
            "importance": model.feature_importances_[:n]
        }).sort_values("importance", ascending=False)

        imp_df.to_csv(
            os.path.join(IMPORTANCE_PATH, f"feature_importance_{target}.csv"),
            index=False
        )

        print("\nTop features:", target)
        display(imp_df.head(25))

best_summary_df = pd.DataFrame(summary_rows)
best_summary_df.to_csv(os.path.join(MODEL_PATH, "best_model_summary.csv"), index=False)

display(best_summary_df.sort_values("RMSE"))



Top features: glucose_60m


,target,feature,importance
304,glucose_60m,ts_glucose_mean_30m,0.126843
307,glucose_60m,ts_glucose_max_30m,0.072020
309,glucose_60m,ts_glucose_last_30m,0.026446
348,glucose_60m,ts_prev_meal_auc_2h,0.025806
17,glucose_60m,premeal_glucose,0.021232
242,glucose_60m,activity_additional_physical_activity_data_min...,0.012841
332,glucose_60m,ts_glucose_std_120m,0.011277
326,glucose_60m,ts_glucose_range_90m,0.009902
329,glucose_60m,ts_glucose_slope_90m,0.009868
344,glucose_60m,ts_time_since_prev_meal_hr,0.008046



Top features: glucose_90m


,target,feature,importance
348,glucose_90m,ts_prev_meal_auc_2h,0.262875
304,glucose_90m,ts_glucose_mean_30m,0.121154
307,glucose_90m,ts_glucose_max_30m,0.079470
344,glucose_90m,ts_time_since_prev_meal_hr,0.045149
343,glucose_90m,ts_time_since_prev_meal_min,0.044476
345,glucose_90m,ts_prev_meal_glucose_60m,0.023666
327,glucose_90m,ts_glucose_last_90m,0.020021
318,glucose_90m,ts_glucose_last_60m,0.018442
20,glucose_90m,premeal_mean_30m,0.015618
35,glucose_90m,time_to_next_meal_min,0.015359



Top features: glucose_120m


...


Top features: delta_peak_2h


,target,feature,importance
304,delta_peak_2h,ts_glucose_mean_30m,0.126843
307,delta_peak_2h,ts_glucose_max_30m,0.072020
309,delta_peak_2h,ts_glucose_last_30m,0.026446
348,delta_peak_2h,ts_prev_meal_auc_2h,0.025806
17,delta_peak_2h,premeal_glucose,0.021232
242,delta_peak_2h,activity_additional_physical_activity_data_min...,0.012841
332,delta_peak_2h,ts_glucose_std_120m,0.011277
326,delta_peak_2h,ts_glucose_range_90m,0.009902
329,delta_peak_2h,ts_glucose_slope_90m,0.009868
344,delta_peak_2h,ts_time_since_prev_meal_hr,0.008046



Top features: auc_2h


,target,feature,importance
348,auc_2h,ts_prev_meal_auc_2h,0.262875
304,auc_2h,ts_glucose_mean_30m,0.121154
307,auc_2h,ts_glucose_max_30m,0.079470
344,auc_2h,ts_time_since_prev_meal_hr,0.045149
343,auc_2h,ts_time_since_prev_meal_min,0.044476
345,auc_2h,ts_prev_meal_glucose_60m,0.023666
327,auc_2h,ts_glucose_last_90m,0.020021
318,auc_2h,ts_glucose_last_60m,0.018442
20,auc_2h,premeal_mean_30m,0.015618
35,auc_2h,time_to_next_meal_min,0.015359


,target,best_model,MAE,RMSE,R2,model_file
4,delta_peak_2h,XGBoost,6.771078,10.987536,0.756399,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
3,peak_glucose_2h,XGBoost,6.109855,11.214652,0.658058,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
0,glucose_60m,XGBoost,6.416120,12.296146,0.583307,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
1,glucose_90m,Random Forest,9.333785,12.523727,0.393645,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
2,glucose_120m,CatBoost,11.747674,15.001181,0.187027,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
5,auc_2h,Random Forest,479.910825,900.209086,0.769779,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...


In [16]:
print("Notebook 5 Time-Series complete.")
print("Results saved in:", MODEL_PATH)
print("Compare this with the previous image-enriched results.")
display(best_summary_df)


Notebook 5 Time-Series complete.
Results saved in: C:\Users\Shabnam\Meal_Centric_Glucose_Dissertation\05_regression_models_time_series
Compare this with the previous image-enriched results.


,target,best_model,MAE,RMSE,R2,model_file
0,glucose_60m,XGBoost,6.416120,12.296146,0.583307,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
1,glucose_90m,Random Forest,9.333785,12.523727,0.393645,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
2,glucose_120m,CatBoost,11.747674,15.001181,0.187027,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
3,peak_glucose_2h,XGBoost,6.109855,11.214652,0.658058,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
4,delta_peak_2h,XGBoost,6.771078,10.987536,0.756399,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
5,auc_2h,Random Forest,479.910825,900.209086,0.769779,C:\Users\Shabnam\Meal_Centric_Glucose_Disserta...
